1

In [5]:
import json
import re
import random
from collections import defaultdict
from google.colab import files

SEED = 42
random.seed(SEED)

sql_filename = "/content/conversations_dump.sql"
s2_filename  = "/content/strategy2_finetune.jsonl"

import os
if not os.path.exists(sql_filename):
    raise FileNotFoundError(f"{sql_filename} not found — check the sidebar filename")
if not os.path.exists(s2_filename):
    raise FileNotFoundError(f"{s2_filename} not found — check the sidebar filename")

with open(s2_filename) as f:
    strategy2_examples = [json.loads(l) for l in f]

print(f"✅ SQL dump  : {sql_filename}")
print(f"✅ Strategy2 : {s2_filename}")
print(f"   Existing strategy2 examples: {len(strategy2_examples)}")

✅ SQL dump  : /content/conversations_dump.sql
✅ Strategy2 : /content/strategy2_finetune.jsonl
   Existing strategy2 examples: 1088


2

In [6]:
S1_COLS = {
    "id": 0, "task_id": 1, "personality": 2, "problem_text": 3,
    "test_cases": 4, "conversation": 5, "execution_result": 6,
    "solved": 7, "tests_passed": 8, "total_tests": 9, "turns": 10,
    "timestamp": 11, "date_folder": 12, "quality_bucket": 13,
    "model_used": 14, "created_at": 15, "discarded": 16,
}
S1_EXPECTED_COLS = 17

print("\n🔍 Parsing SQL dump...")
with open(sql_filename, encoding="utf-8", errors="replace") as f:
    sql_lines = f.readlines()

s1_start = None
for i, line in enumerate(sql_lines):
    if line.startswith("COPY public.strategy1_conversations "):
        s1_start = i + 1
        break
if s1_start is None:
    raise ValueError("Could not find strategy1_conversations COPY block")

s1_rows = []
for line in sql_lines[s1_start:]:
    stripped = line.strip()
    if stripped.startswith("COPY ") or stripped.startswith("--"):
        break
    if not stripped or stripped == r"\.":
        continue
    if stripped.count("\t") == S1_EXPECTED_COLS - 1:
        s1_rows.append(stripped)

print(f"   Raw strategy1 rows: {len(s1_rows)}")


🔍 Parsing SQL dump...
   Raw strategy1 rows: 2995


3

In [7]:
def safe_json(raw: str):
    """Parse JSON safely, return None on failure."""
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return None


def looks_like_python(text: str) -> bool:
    """
    Heuristic: does this string contain Python code?
    Needs at least 2 structural signals to avoid false positives
    on plain English tutor messages.
    """
    if not text or len(text.strip()) < 20:
        return False
    signals = [
        r"\bdef\s+\w+\s*\(",
        r"\breturn\b",
        r"\bfor\s+\w+\s+in\b",
        r"\bif\s+.+:",
        r"\bwhile\s+.+:",
        r"^\s{2,}",
    ]
    matches = sum(1 for s in signals if re.search(s, text, re.MULTILINE))
    return matches >= 2


def extract_code(content: str) -> str | None:
    """
    Extract Python code from a tutor turn.
    Handles: pure code, ```python ... ``` blocks, or def blocks in mixed text.
    """
    content = content.replace("\\r\\n", "\n").replace("\\n", "\n").strip()

    if looks_like_python(content):
        return content

    code_block = re.search(r"```(?:python)?\s*(.*?)```", content, re.DOTALL)
    if code_block:
        c = code_block.group(1).strip()
        if looks_like_python(c):
            return c

    def_match = re.search(r"(def\s+\w+\s*\(.*?)(?:\n\n|\Z)", content, re.DOTALL)
    if def_match:
        c = def_match.group(1).strip()
        if looks_like_python(c):
            return c

    return None


def get_tutor_code_turns(conversation: list) -> list[str]:
    """Return all tutor code outputs in order, deduped (consecutive identical removed)."""
    codes = []
    for msg in conversation:
        if msg.get("role") != "tutor":
            continue
        code = extract_code(msg.get("content", ""))
        if code and (not codes or code != codes[-1]):
            codes.append(code)
    return codes


4

In [8]:
PERSONALITY_DESCRIPTIONS = {
    "CONFUSED_STUDENT": (
        "a confused student who misunderstands the problem requirements "
        "and writes code that doesn't quite match what's being asked"
    ),
    "SYNTAX_STRUGGLER": (
        "a student who struggles with Python syntax and makes errors like "
        "wrong indentation, missing colons, or incorrect operator usage"
    ),
    "IMPATIENT_STUDENT": (
        "an impatient student who writes code quickly without thinking it "
        "through, often missing edge cases or making off-by-one errors"
    ),
    "PROGRAMMING_HELPER": (
        "a well-meaning but inexperienced helper who writes mostly-correct "
        "code but introduces subtle logical bugs"
    ),
    "OVERCONFIDENT_WRONG": (
        "an overconfident student who writes code that looks polished and "
        "complete but contains fundamental logical errors"
    ),
}


5

In [9]:
SYSTEM_PROMPT = (
    "You are an AI tutoring assistant for an Intelligent Tutoring System. "
    "When given a [NEGATIVE] prompt, generate Python code that contains bugs "
    "appropriate to the specified student persona or bug type. "
    "When given a [POSITIVE] prompt, generate correct, clean Python code."
)


def build_personality_negative_prompt(problem: str, personality: str) -> str:
    """NEGATIVE prompt conditioned on student personality (strategy1)."""
    desc = PERSONALITY_DESCRIPTIONS.get(
        personality,
        f"a {personality.lower().replace('_', ' ')} student"
    )
    return (
        f"[NEGATIVE] Generate a Python function for the following problem "
        f"as it would be written by {desc}.\n\n"
        f"Problem: {problem.strip()}"
    )


def build_positive_prompt(problem: str) -> str:
    return (
        f"[POSITIVE] Generate a correct Python function for the following problem.\n\n"
        f"Problem: {problem.strip()}"
    )


def make_example(user_prompt: str, code: str) -> dict:
    return {
        "conversations": [
            {"role": "user",      "content": user_prompt},
            {"role": "assistant", "content": code},
        ]
    }


print("\n⚙️  Extracting strategy1 personality-based examples...")

strategy1_examples = []
stats = defaultdict(int)
per_personality = defaultdict(lambda: {"neg": 0, "pos": 0})

for row in s1_rows:
    cols = row.split("\t")
    if len(cols) != S1_EXPECTED_COLS:
        stats["wrong_col_count"] += 1
        continue

    problem     = cols[S1_COLS["problem_text"]]
    personality = cols[S1_COLS["personality"]]
    solved      = cols[S1_COLS["solved"]] == "t"
    discarded   = cols[S1_COLS["discarded"]] == "t"

    if discarded:
        stats["discarded"] += 1
        continue
    if not problem or not personality:
        stats["missing_problem_or_personality"] += 1
        continue

    conversation = safe_json(cols[S1_COLS["conversation"]])
    if not conversation:
        stats["bad_json"] += 1
        continue

    tutor_codes = get_tutor_code_turns(conversation)
    if not tutor_codes:
        stats["no_code_turns"] += 1
        continue

    # Filter out stub-only first attempts (e.g. "def f(x): pass")
    if len(tutor_codes[0].strip()) < 30:
        stats["first_attempt_too_short"] += 1
        continue

    first_attempt = tutor_codes[0]
    final_code    = tutor_codes[-1]
    one_shot      = (first_attempt == final_code)

    # ── Decide what training signal this row provides ──────────────────────
    #
    # Case A: Solved in one shot (first == last == correct)
    #   → First attempt is CORRECT code, not buggy
    #   → Add as POSITIVE only. No NEGATIVE (this persona didn't bug it).
    #
    # Case B: Solved after iteration (first != last, solved=true)
    #   → First attempt has bugs that persona introduced → NEGATIVE
    #   → Last attempt is correct → POSITIVE
    #
    # Case C: Never solved (solved=false)
    #   → First attempt has bugs → NEGATIVE only
    #   → No correct code available → no POSITIVE
    # ───────────────────────────────────────────────────────────────────────

    if one_shot and solved:
        # Case A: first attempt IS the correct answer
        pos_prompt = build_positive_prompt(problem)
        strategy1_examples.append(make_example(pos_prompt, first_attempt))
        per_personality[personality]["pos"] += 1
        stats["positive_added_from_one_shot"] += 1

    else:
        # Cases B and C: first attempt is buggy → NEGATIVE
        neg_prompt = build_personality_negative_prompt(problem, personality)
        strategy1_examples.append(make_example(neg_prompt, first_attempt))
        per_personality[personality]["neg"] += 1
        stats["negative_added"] += 1

        # Case B only: also add the final correct version
        if solved and final_code != first_attempt:
            pos_prompt = build_positive_prompt(problem)
            strategy1_examples.append(make_example(pos_prompt, final_code))
            per_personality[personality]["pos"] += 1
            stats["positive_added_from_fix"] += 1

print(f"\n   Strategy1 examples added: {len(strategy1_examples)}")
print(f"\n   Per personality breakdown:")
print(f"   {'PERSONALITY':25s} {'NEG':>6s} {'POS':>6s}")
for p, counts in per_personality.items():
    print(f"   {p:25s} {counts['neg']:>6d} {counts['pos']:>6d}")

print(f"\n   Skipped rows:")
for reason, count in stats.items():
    if reason in ("negative_added", "positive_added"):
        continue
    print(f"     {reason:40s} {count}")



⚙️  Extracting strategy1 personality-based examples...

   Strategy1 examples added: 3204

   Per personality breakdown:
   PERSONALITY                  NEG    POS
   CONFUSED_STUDENT             255    407
   SYNTAX_STRUGGLER             252    383
   IMPATIENT_STUDENT            258    361
   PROGRAMMING_HELPER           287    354
   OVERCONFIDENT_WRONG          287    360

   Skipped rows:
     positive_added_from_fix                  635
     positive_added_from_one_shot             1230
     bad_json                                 425
     first_attempt_too_short                  1


6

In [10]:
combined = strategy2_examples + strategy1_examples
random.shuffle(combined)

print(f"\n📊 Final combined dataset:")
print(f"   Strategy2 (bug-type conditioned): {len(strategy2_examples)}")
print(f"   Strategy1 (personality conditioned): {len(strategy1_examples)}")
print(f"   Total examples: {len(combined)}")

neg = sum(1 for e in combined if "[NEGATIVE]" in e["conversations"][0]["content"])
pos = sum(1 for e in combined if "[POSITIVE]" in e["conversations"][0]["content"])
print(f"   NEGATIVE: {neg}")
print(f"   POSITIVE: {pos}")

out_path = "combined_personality_finetune.jsonl"
with open(out_path, "w") as f:
    for ex in combined:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print(f"\n✅ Saved → {out_path}")
files.download(out_path)


📊 Final combined dataset:
   Strategy2 (bug-type conditioned): 1088
   Strategy1 (personality conditioned): 3204
   Total examples: 4292
   NEGATIVE: 1883
   POSITIVE: 2409

✅ Saved → combined_personality_finetune.jsonl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

7

In [11]:
print("\n── Sample NEGATIVE examples for each personality ──────────────────")
seen = set()
for ex in combined:
    prompt = ex["conversations"][0]["content"]
    if "[NEGATIVE]" not in prompt:
        continue
    for p in PERSONALITY_DESCRIPTIONS:
        desc = PERSONALITY_DESCRIPTIONS[p]
        if desc in prompt and p not in seen:
            print(f"\n=== {p} ===")
            print("PROMPT:", prompt[:200], "...")
            print("CODE:")
            print(ex["conversations"][1]["content"][:300])
            seen.add(p)
            break
    if len(seen) == len(PERSONALITY_DESCRIPTIONS):
        break



── Sample NEGATIVE examples for each personality ──────────────────

=== IMPATIENT_STUDENT ===
PROMPT: [NEGATIVE] Generate a Python function for the following problem as it would be written by an impatient student who writes code quickly without thinking it through, often missing edge cases or making o ...
CODE:
def get_Position(arr, n, x):
    return n - 1

=== SYNTAX_STRUGGLER ===
PROMPT: [NEGATIVE] Generate a Python function for the following problem as it would be written by a student who struggles with Python syntax and makes errors like wrong indentation, missing colons, or incorre ...
CODE:
def count_Rectangles(r):
    return 4*r

=== CONFUSED_STUDENT ===
PROMPT: [NEGATIVE] Generate a Python function for the following problem as it would be written by a confused student who misunderstands the problem requirements and writes code that doesn't quite match what's ...
CODE:
def get_noOfways(n):
    if n == 1:
        return 1
    elif n == 2:
        return 2
    else:
        retu